In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/abdelrahmansh/test-file/test_report.pdf


In [2]:
 !pip install pymupdf pillow python-docx transformers accelerate qwen-vl-utils -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 58.8 MB/s eta 0:00:00


## Imports

In [3]:
import os
import io
import sys
import json
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import fitz  
from PIL import Image
try:
    import docx
except ImportError:
    docx = None

## Your Inputs

In [4]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/abdelrahmansh
/kaggle/input/datasets/abdelrahmansh/test-file


## Vision-language model

In [5]:
_model = None
_processor = None

MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"


def _load_model():
    """Load Qwen2-VL once. Requires a GPU runtime (Colab/Kaggle T4)."""
    global _model, _processor
    if _model is not None:
        return

    

    print(f"Loading {MODEL_NAME} (first run only, ~1-2 min)...")
    _model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    _processor = AutoProcessor.from_pretrained(MODEL_NAME)


def caption_image(image: Image.Image, prompt: str = None) -> str:
    """Generate a text description of a single image using Qwen2-VL."""
    _load_model()

    if prompt is None:
        prompt = (
            "Describe this image in 1-3 clear sentences. If it contains "
            "a chart, table, or diagram, summarize the key information "
            "it conveys rather than just its visual style."
        )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text_prompt = _processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = _processor(text=[text_prompt], images=[image], return_tensors="pt")
    inputs = inputs.to(_model.device)

    generated = _model.generate(**inputs, max_new_tokens=150)
    trimmed = generated[:, inputs.input_ids.shape[1]:]
    output = _processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )
    return output[0].strip()


## PDF handling: text + embedded images, page by page

In [6]:
def _process_pdf(path: str, min_image_size: int = 100) -> dict:
    doc = fitz.open(path)
    full_text_parts = []
    images_out = []

    for page_number in range(len(doc)):
        page = doc.load_page(page_number)

        page_text = page.get_text().strip()
        if page_text:
            full_text_parts.append(page_text)

        for img_index, img in enumerate(page.get_images(full=True)):
            xref = img[0]
            try:
                base_image = doc.extract_image(xref)
                img_bytes = base_image["image"]
                pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            except Exception as e:
                print(f"  [warn] could not extract image on page {page_number+1}: {e}")
                continue

            # skip tiny images (icons, bullets, decorative artifacts)
            if pil_img.width < min_image_size or pil_img.height < min_image_size:
                continue

            try:
                caption = caption_image(pil_img)
            except Exception as e:
                caption = f"[captioning failed: {e}]"

            images_out.append({
                "page": page_number + 1,
                "index": img_index,
                "caption": caption,
            })

    doc.close()
    return {
        "source": path,
        "text": "\n\n".join(full_text_parts).strip(),
        "images": images_out,
    }

## Word (.docx) handling: text + embedded images

In [7]:
def _process_docx(path: str, min_image_size: int = 100) -> dict:
    if docx is None:
        raise ImportError("python-docx not installed. Run: pip install python-docx")

    document = docx.Document(path)

    text_parts = [p.text for p in document.paragraphs if p.text.strip()]

    images_out = []
    img_index = 0
    for rel in document.part.rels.values():
        if "image" in rel.reltype:
            try:
                img_bytes = rel.target_part.blob
                pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            except Exception as e:
                print(f"  [warn] could not extract an embedded image: {e}")
                continue

            if pil_img.width < min_image_size or pil_img.height < min_image_size:
                continue

            try:
                caption = caption_image(pil_img)
            except Exception as e:
                caption = f"[captioning failed: {e}]"

            images_out.append({
                "page": None,  # docx has no fixed page concept
                "index": img_index,
                "caption": caption,
            })
            img_index += 1

    return {
        "source": path,
        "text": "\n\n".join(text_parts).strip(),
        "images": images_out,
    }

## Standalone image file handling

In [8]:
def _process_image(path: str) -> dict:
    pil_img = Image.open(path).convert("RGB")
    caption = caption_image(pil_img)
    return {
        "source": path,
        "text": "",
        "images": [{"page": None, "index": 0, "caption": caption}],
    }

## manual text insertion

In [9]:

def _process_text(path: str) -> dict:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()
    return {
        "source": path,
        "text": text.strip(),
        "images": [],
    }

## Public entry point

In [10]:
def process_file(path: str) -> dict:
    ext = os.path.splitext(path)[1].lower()

    if ext == ".pdf":
        return _process_pdf(path)
    elif ext == ".docx":
        return _process_docx(path)
    elif ext == ".txt":
        return _process_text(path)
    elif ext in (".png", ".jpg", ".jpeg", ".webp"):
        return _process_image(path)
    else:
        raise ValueError(f"Unsupported file type: {ext}")

In [11]:
def merge_output(result: dict) -> dict:
    """
    Combine text + image captions into ONE unified text block, ready to
    hand off to the next stage (embedding / detection model).

    Image captions are inserted inline, tagged with their page number,
    so the downstream model gets full context in reading order instead
    of two disconnected fields.
    """
    parts = [result["text"]]

    for img in result["images"]:
        page_tag = f"page {img['page']}" if img["page"] else "image"
        parts.append(f"[Image on {page_tag}]: {img['caption']}")

    merged_text = "\n\n".join(p for p in parts if p.strip())

    return {
        "source": result["source"],
        "merged_text": merged_text,
        "images": result["images"], 
    }

In [12]:
if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python file_handler.py <folder_of_sample_files>")
        sys.exit(1)

    folder = "/kaggle/input/datasets/abdelrahmansh/test-file"
    files = [
        os.path.join(folder, f) for f in sorted(os.listdir(folder))
        if os.path.splitext(f)[1].lower() in (".pdf", ".docx", ".png", ".jpg", ".jpeg", ".webp")
    ]

    if not files:
        print(f"No supported files found in {folder}")
        sys.exit(1)

    print(f"Found {len(files)} file(s). Processing...")
    results = []
    for f in files:
        print(f"\n--- {f} ---")
        try:
            r = process_file(f)
            merged = merge_output(r)
            print(f"Text length: {len(r['text'])} chars | Images captioned: {len(r['images'])}")
            for img in r["images"]:
                print(f"  page {img['page']}: {img['caption'][:120]}")
            results.append(merged)
        except Exception as e:
            print(f"FAILED: {e}")

    # This is the file you hand off to your teammate's model — one JSON
    # array, each entry with a single merged_text string ready to embed.
    out_path = os.path.join("/kaggle/working", "results.json")

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"Saved merged results to {out_path}")

Found 1 file(s). Processing...

--- /kaggle/input/datasets/abdelrahmansh/test-file/test_report.pdf ---
Loading Qwen/Qwen2-VL-2B-Instruct (first run only, ~1-2 min)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Text length: 105 chars | Images captioned: 1
  page 1: This image is a bar chart titled "Quarterly Revenue Chart." The chart shows three bars representing different quarters. 
Saved merged results to /kaggle/working/results.json
